In [0]:
%sql
SELECT cief.external_id, REPLACE(TRIM(LOWER(REPLACE(cief.brand_name, '_', ','))), '’', '') AS brand
, CASE WHEN cief.brand_name = '[TBD]' THEN 1 ELSE 0 END AS brand_tbd

, cief.source
FROM (
SELECT external_id, COUNT(DISTINCT brand_name) AS dist_brand
FROM prod.detection.commercial_id_external_firehose
WHERE fk_client_id = 753
GROUP BY 1) x
JOIN prod.detection.commercial_id_external_firehose cief
  ON cief.external_id = x.external_id
WHERE fk_client_id = 753
AND dist_brand > 1
-- GROUP BY ALL
ORDER BY 1, rn

In [0]:
%sql
SELECT CURRENT_DATE

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.dma_bin

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.dma_regions;
CREATE TABLE dev.mohit_gangwani.dma_regions AS
SELECT dma_id, alt_region_id AS region_id
FROM dev.mohit_gangwani.dma_bin
GROUP BY 1, 2

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_filtered_set;
CREATE TABLE dev.mohit_gangwani.ad_labeling_filtered_set AS
SELECT vc.external_id AS ad_id
, brand
, NVL(vc.fk_dma_id, 0) AS fk_dma_id
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN prod.detection.location l
  ON l.location_id = vc.fk_location_id
 AND l.country_code = 'US'
JOIN (
  SELECT cief.fk_commercial_id, cief.external_id
  , CASE WHEN NULLIF(cief.brand_name, '[TBD]') IS NULL THEN TRIM(LOWER(NULLIF(cief.title, '[TBD]')))
         WHEN LOWER(NULLIF(cief.title, '[TBD]')) IS NULL THEN TRIM(LOWER(REPLACE(NULLIF(cief.brand_name, '[TBD]'), '_', ',')))
         ELSE TRIM(LOWER(REPLACE(NULLIF(cief.brand_name, '[TBD]'), '_', ',')))||'-'||LOWER(NULLIF(cief.title, '[TBD]')) END AS brand
  , ROW_NUMBER() OVER (PARTITION BY cief.external_id ORDER BY CASE WHEN cief.brand_name = '[TBD]' THEN 1 ELSE 0 END, cief.source DESC, LEN(cief.brand_name)) AS rn
  FROM prod.detection.commercial_id_external_firehose cief
  JOIN prod.detection.clients cl
    ON cl.client_id = cief.fk_client_id
  --WHERE cl.client_name = 'kinetiq'
) cief
  ON cief.external_id = vc.external_id
 AND cief.rn = 1
WHERE vc.session_start >= '2026-02-10 00:00:00'
  AND vc.session_start < '2026-02-17 00:00:00'
  AND vc.fk_zoo_id = 17
GROUP BY 1, 2, 3
HAVING COUNT(DISTINCT vc.fk_tvid||'_'||vc.session_start) >= 20
ORDER BY 1
;
OPTIMIZE dev.mohit_gangwani.ad_labeling_filtered_set
ZORDER BY (ad_id);

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_base_viewing_table_new;
CREATE TABLE dev.mohit_gangwani.ad_labeling_base_viewing_table_new AS
SELECT vc.fk_tvid
, vc.session_start
, COALESCE(vc.external_id, cief.ad_id) AS ad_id
, dma.region_id
, cief.brand
, vc.fk_dma_id AS fk_dma_id
FROM prod.detection.viewing_commercials_firehose_dedup vc
JOIN dev.mohit_gangwani.ad_labeling_filtered_set cief
  ON cief.ad_id = vc.external_id
 AND cief.fk_dma_id = NVL(vc.fk_dma_id, 0)
JOIN prod.detection.location l
  ON l.location_id = vc.fk_location_id
 AND l.country_code = 'US'
LEFT JOIN dev.mohit_gangwani.dma_regions dma
  ON dma.dma_id = vc.fk_dma_id
WHERE vc.session_start >= '2026-02-10 00:00:00'
  AND vc.session_start < '2026-02-17 00:00:00'
  AND vc.fk_zoo_id = 17
GROUP BY ALL
ORDER BY 3, 2, 1;

OPTIMIZE dev.mohit_gangwani.ad_labeling_base_viewing_table_new
ZORDER BY (ad_id, session_start);

In [0]:
%sql
-- # Active TVs per DMA × 10-min bin × platform/station
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_opportunities_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_opportunities_unbinned AS
WITH total_count AS (
  SELECT COUNT(DISTINCT fk_tvid) AS ttl_tvs
  FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_new
)
SELECT a.fk_dma_id
, COUNT(DISTINCT a.fk_tvid) AS active_tvs
, COUNT(DISTINCT a.fk_tvid||'-'||session_start) AS total_impressions
, active_tvs*1.0/t.ttl_tvs AS dma_perc
FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_new a, total_count t
GROUP BY 1, t.ttl_tvs;

SELECT * FROM dev.mohit_gangwani.ad_labeling_opportunities_unbinned

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_impression_count_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_impression_count_unbinned AS
SELECT n.fk_dma_id
, n.ad_id
, COUNT(DISTINCT fk_tvid||'_'||session_start) AS impressions
FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_new n
GROUP BY 1,2;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_dma_base_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_dma_base_unbinned AS
SELECT i.ad_id
, i.fk_dma_id
, i.impressions
, o.active_tvs
, (i.impressions * 1.0) / o.active_tvs AS rate_per_tv
FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned i
LEFT JOIN dev.mohit_gangwani.ad_labeling_opportunities_unbinned o
  ON i.fk_dma_id = o.fk_dma_id
GROUP BY ALL;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_dma_perc_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_dma_perc_unbinned AS
WITH ad_dma AS (
  SELECT ad_id
  , fk_dma_id
  , impressions AS impression_count
  FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned
)
, ad_tot AS (
  SELECT ad_id
  , SUM(impressions) AS total_impressions
  FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned
  GROUP BY ad_id

)
, joined AS (
SELECT d.ad_id
, d.fk_dma_id
, d.impression_count
, t.total_impressions
, o.active_tvs
, o.dma_perc
, d.impression_count/t.total_impressions AS per_dma
FROM ad_dma d
JOIN ad_tot t
  ON d.ad_id = t.ad_id
JOIN dev.mohit_gangwani.ad_labeling_opportunities_unbinned o
  ON o.fk_dma_id = d.fk_dma_id
)
SELECT *
FROM (
  SELECT *
  , DENSE_RANK() OVER (PARTITION BY ad_id ORDER BY per_dma DESC) AS dma_rank
  , SUM(per_dma) OVER (PARTITION BY ad_id ORDER BY per_dma DESC) AS cumulative_perc
  FROM joined
)
WHERE dma_rank = 1 OR cumulative_perc <= 0.90

In [0]:
%sql
SELECT dma_count, COUNT(DISTINCT ad_id)
FROM (
SELECT ad_id
, COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
FROM dev.mohit_gangwani.ad_labeling_dma_perc_unbinned
GROUP BY 1)
GROUP BY 1
ORDER BY 1 DESC

In [0]:
%sql
SELECT AVG(dma_count)
FROM (
SELECT ad_id
, COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
FROM dev.mohit_gangwani.ad_labeling_dma_perc_unbinned
GROUP BY 1)
-- WHERE dma_count > 1
-- GROUP BY 1
ORDER BY 1 DESC

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_over_90_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_over_90_unbinned AS
SELECT ad_id
, COUNT(DISTINCT fk_dma_id)*1.0 AS dma_in_90
FROM dev.mohit_gangwani.ad_labeling_dma_perc_unbinned
GROUP BY 1

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_coverage_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_coverage_unbinned AS
WITH flags AS (
  SELECT ad_id
  , COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
  FROM dev.mohit_gangwani.ad_labeling_dma_base_unbinned
  GROUP BY 1
)
, ttl AS (
  SELECT COUNT(DISTINCT fk_dma_id)*1.0 AS dma_count
  FROM dev.mohit_gangwani.ad_labeling_dma_base_unbinned
)
SELECT a.ad_id
, a.dma_count/t.dma_count AS coverage_score
FROM flags a, ttl t
GROUP BY ALL;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_entropy_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_entropy_unbinned AS
-- 1) DMA universe (this is what adds the "0s")
WITH dma_universe AS (
  SELECT fk_dma_id
  , active_tvs
  FROM dev.mohit_gangwani.ad_labeling_opportunities_unbinned
  GROUP BY ALL
)
-- 2) Ad universe
, ad_universe AS (
  SELECT DISTINCT ad_id
  FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned
  GROUP BY ALL
)
-- 3) Full grid: every (ad) × every DMA
, grid AS (
  SELECT a.ad_id
  , d.fk_dma_id
  , d.active_tvs
  FROM ad_universe a, dma_universe d
  GROUP BY ALL
)
-- 4) Left-join impressions; missing DMAs become 0 impressions
, joined AS (
  SELECT g.ad_id
  , g.fk_dma_id
  , g.active_tvs
  , COALESCE(i.impressions, 0) AS impressions
  FROM grid g
  LEFT JOIN dev.mohit_gangwani.ad_labeling_impression_count_unbinned i
    ON g.ad_id = i.ad_id
   AND g.fk_dma_id = i.fk_dma_id
  GROUP BY ALL
)
-- 5) Convert to TV-normalized "intensity" (imps per active TV)
, rates AS (
  SELECT ad_id
  , fk_dma_id
  , CASE WHEN active_tvs > 0 THEN impressions * 1.0 / active_tvs
         ELSE 0.0
    END AS rate_per_tv
  FROM joined
)
-- 6) Total mass per (ad) for probability normalization
, tot AS (
  SELECT ad_id
  , SUM(rate_per_tv) AS total_rate
  , COUNT(*) AS k_total_dmas
  FROM rates
  GROUP BY 1
)
-- 7) Probability distribution over ALL DMAs (includes zeros)
, p AS (
  SELECT r.ad_id
  , r.fk_dma_id
  , t.k_total_dmas
  , CASE WHEN t.total_rate > 0 THEN r.rate_per_tv / t.total_rate
         ELSE 0.0
    END AS p_dma
  FROM rates r
  JOIN tot t
    ON r.ad_id = t.ad_id
)
SELECT ad_id
, MAX(k_total_dmas) AS k_total_dmas
, -SUM(CASE WHEN p_dma > 0 THEN p_dma * LOG(p_dma) ELSE 0 END) AS entropy
, CASE WHEN MAX(k_total_dmas) > 1 THEN (-SUM(CASE WHEN p_dma > 0 THEN p_dma * LOG(p_dma) ELSE 0 END)) / LOG(MAX(k_total_dmas))
       ELSE 0.0
  END AS entropy_norm
FROM p
GROUP BY 1;ohi

In [0]:
%sql
-- significant_dma_count_05 = count of DMAs where share >= 0.05
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_sig_dma_count_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_sig_dma_count_unbinned AS
WITH base AS (
  SELECT ad_id
  , fk_dma_id
  , impressions
  FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned
),
tot AS (
  SELECT ad_id
  , SUM(impressions) * 1.0 AS tot_imp
  FROM base
  GROUP BY 1
),
shares AS (
  SELECT b.ad_id
  , b.fk_dma_id
  , b.impressions * 1.0 / t.tot_imp AS dma_share
  FROM base b
  JOIN tot t
  ON b.ad_id = t.ad_id
)
SELECT ad_id
, SUM(CASE WHEN dma_share >= 0.05 THEN 1 ELSE 0 END) AS significant_dma_count_05
FROM shares
GROUP BY 1;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_dma_mix_ratio_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_dma_mix_ratio_unbinned AS
WITH base AS (
  SELECT ad_id
  , fk_dma_id
  , impressions
  , ROW_NUMBER() OVER (PARTITION BY ad_id ORDER BY impressions DESC) AS rn
  FROM dev.mohit_gangwani.ad_labeling_impression_count_unbinned
)
, tot AS (
  SELECT ad_id
  , SUM(impressions) * 1.0 AS tot_imp
  FROM base
  GROUP BY 1
),
top5 AS (
  SELECT ad_id
  , SUM(impressions) * 1.0 AS top5_imp
  FROM base
  WHERE rn <= 5
  GROUP BY 1
)
SELECT t.ad_id
, CASE WHEN t.tot_imp > 0 THEN top5.top5_imp / t.tot_imp ELSE 0.0
  END AS top5_dma_mix_ratio
FROM tot t
JOIN top5
 ON t.ad_id = top5.ad_id
GROUP BY ALL;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_region_features_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_region_features_unbinned AS
WITH region_rate AS (
  SELECT d.ad_id
  , CASE WHEN r.region_id > 10 THEN 0 ELSE r.region_id END AS region_id
  , SUM(d.rate_per_tv) AS region_rate
  FROM dev.mohit_gangwani.ad_labeling_dma_base_unbinned d
  JOIN dev.mohit_gangwani.dma_regions r
    ON d.fk_dma_id = r.dma_id
  GROUP BY 1,2
),
tot AS (
  SELECT ad_id
  , SUM(region_rate) AS total_region_rate
  FROM region_rate
  GROUP BY 1
),
ranked AS (
  SELECT rr.ad_id
  , rr.region_id
  , CASE WHEN t.total_region_rate > 0 THEN rr.region_rate / t.total_region_rate ELSE 0.0 END AS region_share
  , ROW_NUMBER() OVER (PARTITION BY rr.ad_id ORDER BY rr.region_rate DESC) AS rn
  , SUM(CASE WHEN t.total_region_rate > 0 THEN rr.region_rate / t.total_region_rate ELSE 0.0 END
        ) OVER (PARTITION BY rr.ad_id
                ORDER BY rr.region_rate DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_share
  FROM region_rate rr
  JOIN tot t
    ON rr.ad_id = t.ad_id
)
SELECT ad_id
, MIN(CASE WHEN cum_share >= 0.90 THEN rn END) AS region_count_90
, SUM(CASE WHEN region_share >= 0.05 THEN 1 ELSE 0 END) AS significant_region_count_05
, SUM(CASE WHEN rn <= 1 THEN region_share ELSE 0.0 END) AS top_region_mix
, SUM(CASE WHEN rn <= 2 THEN region_share ELSE 0.0 END) AS top2_region_mix
FROM ranked
GROUP BY 1;

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_ad_totals_unbinned;
CREATE TABLE dev.mohit_gangwani.ad_labeling_ad_totals_unbinned AS
SELECT ad_id
, brand
, COUNT(DISTINCT fk_tvid) AS total_opportunities
, COUNT(DISTINCT fk_tvid||'-'||session_start) AS total_impressions
FROM dev.mohit_gangwani.ad_labeling_base_viewing_table_new
GROUP BY 1, 2

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_final_features_unbinned_020326;
CREATE TABLE dev.mohit_gangwani.ad_labeling_final_features_unbinned_020326 AS
SELECT t.ad_id
, t.brand
, cief.commercial_category
, t.total_impressions
, t.total_opportunities
, c.coverage_score
, e.entropy_norm
, s.significant_dma_count_05
, o.dma_in_90
, m.top5_dma_mix_ratio
, reg.region_count_90
, reg.significant_region_count_05
, reg.top_region_mix
, reg.top2_region_mix
FROM dev.mohit_gangwani.ad_labeling_ad_totals_unbinned t
LEFT JOIN dev.mohit_gangwani.ad_labeling_coverage_unbinned c
  ON t.ad_id = c.ad_id
LEFT JOIN dev.mohit_gangwani.ad_labeling_entropy_unbinned e
  ON t.ad_id = e.ad_id
LEFT JOIN dev.mohit_gangwani.ad_labeling_sig_dma_count_unbinned s
  ON t.ad_id = s.ad_id
LEFT JOIN dev.mohit_gangwani.ad_labeling_dma_mix_ratio_unbinned m
  ON t.ad_id = m.ad_id
LEFT JOIN dev.mohit_gangwani.ad_labeling_over_90_unbinned o
  ON t.ad_id = o.ad_id
LEFT JOIN dev.mohit_gangwani.ad_labeling_region_features_unbinned reg
  ON reg.ad_id = t.ad_id
JOIN (
  SELECT cief.external_id
  , cief.commercial_category
  , ROW_NUMBER() OVER (
    PARTITION BY cief.external_id
    ORDER BY CASE WHEN cief.brand_name = '[TBD]' THEN 1 ELSE 0 END,
             CASE WHEN cief.commercial_category = '[TBD]' THEN 1 ELSE 0 END,
             cief.source DESC,
             LEN(cief.brand_name)
    ) AS rn
  FROM prod.detection.commercial_id_external_firehose cief
  JOIN prod.detection.clients cl
    ON cl.client_id = cief.fk_client_id
  --WHERE cl.client_name = 'kinetiq'
) cief
  ON cief.external_id = t.ad_id
 AND cief.rn = 1
GROUP BY ALL
ORDER BY 1;

OPTIMIZE dev.mohit_gangwani.ad_labeling_final_features_unbinned_020326
ZORDER BY (ad_id);

In [0]:
%sql
SELECT brand, COUNT(*) FROM dev.mohit_gangwani.ad_labeling_filtered_set
-- WHERE brand LIKE '%senate%'
-- WHERE brand LIKE '%dealer%'
-- WHERE brand LIKE '%arizona%' or brand LIKE '%california%' or brand LIKE '%nevada%' or brand LIKE '%oregon%' or brand LIKE '%idaho%' or brand LIKE '%montana%' or brand LIKE '%south dakota%' or brand LIKE '%washington%' or brand LIKE '%wyoming%' or brand LIKE '%colorado%' or brand LIKE '%nebraska%' or brand LIKE '%new mexico%' or brand LIKE '%utah%' or brand LIKE '%arkansas%' or brand LIKE '%illinois%' or brand LIKE '%indiana%' or brand LIKE '%iowa%' or brand LIKE '%kansas%' or brand LIKE '%kentucky%' or brand LIKE '%michigan%' or brand LIKE '%minnesota%' or brand LIKE '%missouri%' or brand LIKE '%north dakota%' or brand LIKE '%ohio%' or brand LIKE '%oklahoma%' or brand LIKE '%pennsylvania%' or brand LIKE '%tennessee%' or brand LIKE '%west virginia%' or brand LIKE '%wisconsin%' or brand LIKE '%louisiana%' or brand LIKE '%mississippi%' or brand LIKE '%texas%' or brand LIKE '%connecticut%' or brand LIKE '%delaware%' or brand LIKE '%district of columbia%' or brand LIKE '%maine%' or brand LIKE '%maryland%' or brand LIKE '%massachusetts%' or brand LIKE '%new hampshire%' or brand LIKE '%new jersey%' or brand LIKE '%new york%' or brand LIKE '%rhode island%' or brand LIKE '%vermont%' or brand LIKE '%virginia%' or brand LIKE '%alabama%' or brand LIKE '%florida%' or brand LIKE '%georgia%' or brand LIKE '%north carolina%' or brand LIKE '%south carolina%' or brand LIKE '%alaska%' or brand LIKE '%hawaii%'
GROUP BY 1
ORDER BY 2 DESC
LIMIT 1000

In [0]:
%sql
SELECT 'meta', COUNT(*) FROM dev.mohit_gangwani.ad_labeling_filtered_set meta
UNION
SELECT 'cov', COUNT(*) FROM dev.mohit_gangwani.ad_labeling_coverage_unbinned c
UNION
SELECT 'ent', COUNT(*) FROM dev.mohit_gangwani.ad_labeling_entropy_unbinned e
UNION
SELECT 'sig', COUNT(*) FROM dev.mohit_gangwani.ad_labeling_sig_dma_count_unbinned s
UNION
SELECT 'mix', COUNT(*) FROM dev.mohit_gangwani.ad_labeling_dma_mix_ratio_unbinned m
UNION
SELECT '90', COUNT(*) FROM dev.mohit_gangwani.ad_labeling_over_90_unbinned o
UNION
SELECT 'reg', COUNT(*) FROM dev.mohit_gangwani.ad_labeling_region_features_unbinned reg

In [0]:
%sql
SELECT LEFT('0200',2)::integer*3600 + RIGHT('0200',2)::integer*60